In [25]:
# ## Imports

import pickle
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import json
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, roc_auc_score,
    recall_score, precision_score, accuracy_score,
)
from xgboost import XGBClassifier

In [ ]:
# ## 1 · Data Splitting

def split_dataset(csv_path, save_prefix=None, test_size=0.15, val_size=0.1765, random_state=42):
    """
    Loads a CSV, creates stratified train/val/test splits by cardio × gender.

    Args:
        csv_path     : path to CSV file
        save_prefix  : if provided, saves to ../data/test_train_val_sets/{prefix}_*.csv
        test_size    : proportion for test set (default 0.2)
        val_size     : proportion of train for val (default 0.2 → ~16% of total)
        random_state : random seed

    Returns:
        train_df, val_df, test_df
    """
    df = pd.read_csv(csv_path)
    df["stratify"] = df["cardio"].astype(str) + "_" + df["gender"].astype(str)

    train_df, test_df = train_test_split(
        df, test_size=test_size, stratify=df["stratify"], random_state=random_state,
    )
    train_df, val_df = train_test_split(
        train_df, test_size=val_size, stratify=train_df["stratify"], random_state=random_state,
    )

    print(f"Dataset    : {csv_path}")
    print(f"Train      : {len(train_df)} ({len(train_df)/len(df)*100:.1f}%)")
    print(f"Val        : {len(val_df)}   ({len(val_df)/len(df)*100:.1f}%)")
    print(f"Test       : {len(test_df)}  ({len(test_df)/len(df)*100:.1f}%)")

    if save_prefix:
        base = "../../data/test_train_val_sets"
        train_df.to_csv(f"{base}/{save_prefix}_train.csv", index=False)
        val_df.to_csv(f"{base}/{save_prefix}_val.csv",     index=False)
        test_df.to_csv(f"{base}/{save_prefix}_test.csv",   index=False)
        print(f"Saved to {base}/{save_prefix}_*.csv")
    sizes = {
    "train": len(train_df),
    "val":   len(val_df),
    "test":  len(test_df)
    }
    with open("../../config/dataset_split_sizes.json", "w") as f:
        json.dump(sizes, f, indent=4)
    print("Saved dataset sizes:", sizes)

    return train_df, val_df, test_df



train_df, val_df, test_df = split_dataset(
    "../../data/processed/cardio_baseline_clean.csv",
    save_prefix="cardio_baseline",
)

Dataset    : ../../data/processed/cardio_baseline_clean.csv
Train      : 47721 (70.0%)
Val        : 10229   (15.0%)
Test       : 10227  (15.0%)
Saved to ../../data/test_train_val_sets/cardio_baseline_*.csv
Saved dataset sizes: {'train': 47721, 'val': 10229, 'test': 10227}


In [29]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

def draw_split_table(train_df, val_df, test_df, output_path="split_validation_table.png"):

    def get_rows(df, name):
        total = len(df)
        rows = []
        for sex, label in [(0, "Female"), (1, "Male")]:
            sex_df = df[df["gender"] == sex]
            n = len(sex_df)
            cvd = (sex_df["cardio"] == 1).sum()
            no_cvd = (sex_df["cardio"] == 0).sum()
            rows.append([
                name, label, n,
                f"{n/total*100:.1f}%",
                cvd, f"{cvd/n*100:.1f}%",
                no_cvd, f"{no_cvd/n*100:.1f}%"
            ])
        return rows

    rows = get_rows(train_df, "Train") + get_rows(val_df, "Val") + get_rows(test_df, "Test")

    col_widths  = [0.08, 0.08, 0.08, 0.09, 0.09, 0.09, 0.10, 0.09]
    row_h       = 0.055
    left_start  = 0.02
    cell_pad    = 0.008
    header_y    = 0.97

    total_w = sum(col_widths)
    fig_w   = total_w * 18
    fig, ax = plt.subplots(figsize=(fig_w, len(rows) * 0.42 + 1.5))
    ax.axis("off")

    def draw_cell(ax, x, y, w, h, text, bold=False, fontsize=9,
                  align='left', bg=None):
        if bg:
            ax.add_patch(mpatches.FancyBboxPatch(
                (x, y - h), w - 0.004, h,
                boxstyle="square,pad=0", linewidth=0.5,
                edgecolor='#aaaaaa', facecolor=bg,
                transform=ax.transAxes, clip_on=False
            ))
        tx = x + cell_pad if align == 'left' else x + w / 2
        ax.text(tx, y - h / 2, text,
                transform=ax.transAxes, fontsize=fontsize,
                fontweight='bold' if bold else 'normal',
                ha=align, va='center', color='#1a1a1a')

    # ── Header row 1 ──
    x = left_start
    draw_cell(ax, x, header_y, col_widths[0], row_h * 2, "Split",       bold=True, bg='#f0f0f0')
    x += col_widths[0]
    draw_cell(ax, x, header_y, col_widths[1], row_h * 2, "Group",       bold=True, bg='#f0f0f0')
    x += col_widths[1]
    draw_cell(ax, x, header_y, col_widths[2], row_h * 2, "N",           bold=True, align='center', bg='#f0f0f0')
    x += col_widths[2]
    draw_cell(ax, x, header_y, col_widths[3], row_h * 2, "% of Set",    bold=True, align='center', bg='#f0f0f0')
    x += col_widths[3]

    cvd_w = sum(col_widths[4:6])
    no_cvd_w = sum(col_widths[6:8])
    draw_cell(ax, x, header_y, cvd_w,    row_h, "CVD",    bold=True, align='center', bg='#dce8f5')
    x += cvd_w
    draw_cell(ax, x, header_y, no_cvd_w, row_h, "No CVD", bold=True, align='center', bg='#d5ead5')

    # ── Header row 2 ──
    sub_y = header_y - row_h
    x = left_start + sum(col_widths[:4])
    for i, lbl in enumerate(["N", "%"]):
        draw_cell(ax, x, sub_y, col_widths[4 + i], row_h, lbl, bold=True, fontsize=8, align='center', bg='#e8eef5')
        x += col_widths[4 + i]
    for i, lbl in enumerate(["N", "%"]):
        draw_cell(ax, x, sub_y, col_widths[6 + i], row_h, lbl, bold=True, fontsize=8, align='center', bg='#e2f0e2')
        x += col_widths[6 + i]

    # ── Data rows ──
    prev_split = None
    for r_idx, row in enumerate(rows):
        y_top = header_y - row_h * 2 - r_idx * row_h
        bg    = '#ffffff' if r_idx % 2 == 0 else '#f9f9f9'
        x     = left_start

        # Only show split name once per group
        split_label = row[0] if row[0] != prev_split else ""
        prev_split  = row[0]

        display_row = [split_label] + row[1:]
        for c_idx, val in enumerate(display_row):
            align = 'left' if c_idx < 2 else 'center'
            draw_cell(ax, x, y_top, col_widths[c_idx], row_h, str(val), fontsize=8, align=align, bg=bg)
            x += col_widths[c_idx]

    # ── Outer border ──
    total_h = row_h * 2 + row_h * len(rows)
    ax.add_patch(mpatches.FancyBboxPatch(
        (left_start, header_y - total_h), total_w, total_h,
        boxstyle="square,pad=0", linewidth=1.2,
        edgecolor='#555555', facecolor='none',
        transform=ax.transAxes, clip_on=False
    ))

    plt.tight_layout()
    plt.savefig(output_path, dpi=180, bbox_inches='tight',
                facecolor='white', edgecolor='none')
    plt.close()
    print(f"Saved → {output_path}")

draw_split_table(train_df, val_df, test_df, output_path="split_validation_table.png")

Saved → split_validation_table.png


In [22]:
# ## 2 · Prepare X / y

X_train = train_df.drop(columns=["cardio", "stratify"])
y_train = train_df["cardio"]

X_val = val_df.drop(columns=["cardio", "stratify"])
y_val = val_df["cardio"]

X_test = test_df.drop(columns=["cardio", "stratify"])
y_test = test_df["cardio"]

female_mask_test = test_df["gender"].values == 0
male_mask_test   = test_df["gender"].values == 1

In [23]:
# ## 3 · Train Baseline Model

model = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    random_state=42,
    early_stopping_rounds=10,
)

model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=True)

with open("../baseline_models/cardio_xgb_baseline_model.pkl", "wb") as f:
    pickle.dump(model, f)

[0]	validation_0-logloss:0.68051
[1]	validation_0-logloss:0.66910
[2]	validation_0-logloss:0.65879
[3]	validation_0-logloss:0.64942
[4]	validation_0-logloss:0.64093
[5]	validation_0-logloss:0.63312
[6]	validation_0-logloss:0.62602
[7]	validation_0-logloss:0.61950
[8]	validation_0-logloss:0.61356
[9]	validation_0-logloss:0.60809
[10]	validation_0-logloss:0.60311
[11]	validation_0-logloss:0.59855
[12]	validation_0-logloss:0.59432
[13]	validation_0-logloss:0.59047
[14]	validation_0-logloss:0.58691
[15]	validation_0-logloss:0.58360
[16]	validation_0-logloss:0.58067
[17]	validation_0-logloss:0.57783
[18]	validation_0-logloss:0.57525
[19]	validation_0-logloss:0.57285
[20]	validation_0-logloss:0.57070
[21]	validation_0-logloss:0.56872
[22]	validation_0-logloss:0.56683
[23]	validation_0-logloss:0.56511
[24]	validation_0-logloss:0.56350
[25]	validation_0-logloss:0.56205
[26]	validation_0-logloss:0.56068
[27]	validation_0-logloss:0.55943
[28]	validation_0-logloss:0.55826
[29]	validation_0-loglos

In [24]:
# Early stopping check
print(f"Best iteration: {model.best_iteration}")
print(f"Best validation loss: {model.best_score:.5f}")

# Serialisation check
with open("../baseline_models/cardio_xgb_baseline_model.pkl", "rb") as f:
    model_reloaded = pickle.load(f)

original_preds = model.predict_proba(X_test)[:, 1]
reloaded_preds = model_reloaded.predict_proba(X_test)[:, 1]

print(f"Predictions identical after reload: {np.allclose(original_preds, reloaded_preds)}")

Best iteration: 176
Best validation loss: 0.53947
Predictions identical after reload: True
